# Inspect sample conditioning captions

The single-image previews use MOJAVE observation 0003+380 from 2007-08-24. Its PA and ridgeline are measured in memory with the preprocessing functions. R01 is that real observation; S01 through S11 are synthetic layout examples.

The two 3 × 4 grids use the same tiles, measures, and field order. A dash keeps the field visible when its value is unavailable. Colors and line styles come from the supplied series configuration.

- SHOW_AXES and SHARED_COLORBAR set grid defaults. The colorbar comparison explicitly shows east/north axes in mas.
- SHOW_SOURCE_NAMES controls source titles. ANNOTATED_PANELS can select particular IDs or be left as None to annotate every image.
- ACTIVE_SERIES, SHOW_RIDGE, and FADE_EXTENT_ARCS control which overlays or styles are requested across the whole grid. The overlay-mix example deliberately varies the marks from tile to tile.

In [ ]:
from dataclasses import dataclass
from io import BytesIO
from shutil import which

import matplotlib as mpl
import numpy as np
from IPython.display import Image, display
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, AnnotationBbox, DrawingArea, HPacker, TextArea, VPacker
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MarkerPath

from radio_diffusion.utils import plotting
from radio_diffusion.utils.plotting import ImageOverlay


SHOW_AXES = True
SHARED_COLORBAR = True
SHOW_SOURCE_NAMES = True
ANNOTATED_PANELS = None  # All panels. A set of IDs can select specific panels.
SERIES = {
    "conditioning": dict(label="Conditioning", short="Cond.", color="#259BC4", linestyle="dashed"),
    "estimate": dict(label="Image estimate", short="Est.", color="#D47E16", linestyle="solid"),
}
ACTIVE_SERIES = tuple(SERIES)
SHOW_RIDGE = True
FADE_EXTENT_ARCS = True
EXTENT_ARC_HALF_WIDTH_DEG = 24.0
RIDGE_STYLE = dict(label="Ridgeline", short="Ridge", color="#C98DE1", linestyle="solid")
PREFER_LATEX = True

CAPTION_RC = {
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "axes.edgecolor": "#788594",
    "axes.linewidth": 0.7,
    "axes.labelcolor": "#26344a",
    "text.color": "#26344a",
    "figure.facecolor": "white",
    "mathtext.fontset": "dejavusans",
    "text.usetex": False,
}
LATEX_RC = {
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
}


def latex_available():
    if not (which("latex") and which("dvipng")):
        return False
    with mpl.rc_context(LATEX_RC):
        figure = Figure()
        figure.text(0.1, 0.5, r"$\mathrm{PA}=110^\circ\quad r=2.85\,\mathrm{mas}$")
        try:
            FigureCanvasAgg(figure).draw()
        except (OSError, RuntimeError):
            return False
    return True


USE_LATEX = PREFER_LATEX and latex_available()
MATH_RC = {**CAPTION_RC, **(LATEX_RC if USE_LATEX else {})}
print(f"Math caption renderer: {'LaTeX' if USE_LATEX else 'Matplotlib MathText'}")

In [ ]:
samples = [
    dict(id="S01", source="J0813+2910", image_pa=121, image_extent=2.85, peak=0.44,
         annotations={"conditioning": (106, 2.55), "estimate": (121, 2.85)}),
    dict(id="S02", source="J1126−0427", image_pa=66, image_extent=1.60, peak=0.18,
         annotations={"conditioning": (66, 1.60)}),
    dict(id="S03", source="J1502+3741", image_pa=295, image_extent=2.40, peak=0.56,
         annotations={"estimate": (295, 2.40)}),
    dict(id="S04", source="J2034−1152", image_pa=205, image_extent=2.25, peak=0.29,
         annotations={"conditioning": (193, None), "estimate": (205, 2.25)}),
]

SIZE = 88
REFERENCE = (SIZE - 1) / 2
PIXEL_SCALE_MAS = 0.086
VIEW_MAS = 3.7
INTENSITY_LIMITS = (-0.001, 0.56)
coordinates = plotting.mojave_coordinates({
    "reference_pixel_xy": [REFERENCE, REFERENCE],
    "pixel_to_east_north_mas": [[-PIXEL_SCALE_MAS, 0], [0, PIXEL_SCALE_MAS]],
})


def demo_image(sample, seed):
    row, column = np.indices((SIZE, SIZE))
    east, north = coordinates.from_pixels(np.stack((column, row), axis=-1)).transpose(2, 0, 1)
    pa = np.deg2rad(sample["image_pa"])
    along = east * np.sin(pa) + north * np.cos(pa)
    across = east * np.cos(pa) - north * np.sin(pa)
    extent = sample["image_extent"]
    forward = np.maximum(along, 0)
    bend = 0.035 * forward**2
    width = 0.105 + 0.045 * forward
    core = np.exp(-0.5 * ((along / 0.20)**2 + (across / 0.14)**2))
    jet = 0.28 * np.exp(-forward / 1.5 - 0.5 * ((across - bend) / width)**2)
    jet *= (along >= 0) * np.clip((extent - along) / 0.42, 0, 1)
    knot = 0.095 * np.exp(
        -0.5 * (((along - extent * 0.63) / 0.22)**2 + ((across - bend) / 0.14)**2)
    )
    noise = np.random.default_rng(seed).normal(0, 0.00025, core.shape)
    return sample["peak"] * (core + jet + knot) + noise


for index in range(5, 12):
    pa = (40 + index * 37) % 360
    extent = 1.45 + 0.19 * (index % 8)
    samples.append(dict(
        id=f"S{index:02d}", source=f"Synthetic {index:02d}",
        image_pa=pa, image_extent=extent, peak=0.17 + 0.07 * (index % 5),
        annotations={"conditioning": ((pa - 11) % 360, extent + 0.25), "estimate": (pa, extent)},
    ))


def synthetic_ridge(sample):
    along = np.linspace(0.15, sample["image_extent"], 17)
    pa = np.deg2rad(sample["image_pa"])
    forward = np.array([np.sin(pa), np.cos(pa)])
    across = np.array([np.cos(pa), -np.sin(pa)])
    return along[:, None] * forward + (0.035 * along**2)[:, None] * across


synthetic_samples = samples
for index, sample in enumerate(synthetic_samples):
    sample["image"] = demo_image(sample, seed=40 + index)
    sample["coordinates"] = coordinates
    sample["ridge"] = synthetic_ridge(sample)

In [ ]:
import warnings

from astropy.io import fits
from astropy.wcs import FITSFixedWarning

from radio_diffusion.data.mojave_metadata import (
    JetOrientationSettings, crop_spatial_geometry, get_noise_jy_per_beam,
    jet_orientation_deg, trace_jet_ridgeline,
)
from radio_diffusion.data.preprocess_mojave import PREPROCESSING_SETTINGS
from radio_diffusion.paths import get_path


REAL_EXAMPLE_FILENAME = "0003+380.u.2007_08_24.icn.fits.gz"
REAL_EXAMPLE_SERIES = "estimate"


def read_mojave_example(filename, series):
    """Measure one native-beam FITS observation with the inspection settings."""
    path = get_path("MOJAVE_FITS_DIR") / filename
    orientation = JetOrientationSettings(inner_mas=1.0, outer_mas=3.0, sigma_cut=3.0)
    with fits.open(path, memmap=False) as hdul, warnings.catch_warnings():
        warnings.simplefilter("ignore", FITSFixedWarning)
        image_hdu = hdul[0]
        noise = get_noise_jy_per_beam(image_hdu, "four_corner")
        pa = jet_orientation_deg(image_hdu, hdul["AIPS CC"], noise, orientation)
        ridge = []
        if np.isfinite(pa):
            ridge, _ = trace_jet_ridgeline(image_hdu, noise, pa, PREPROCESSING_SETTINGS.ridgeline)
        frame = plotting.mojave_coordinates(crop_spatial_geometry(image_hdu.header, left=0, top=0))
        ridge_pixels = np.array([(point["x_pixel"], point["y_pixel"]) for point in ridge])
        return dict(
            id="R01",
            source=filename.split(".", 1)[0],
            image=np.asarray(np.squeeze(image_hdu.data), dtype=np.float32).copy(),
            coordinates=frame,
            noise_rms_jy_per_beam=noise,
            ridge=frame.from_pixels(ridge_pixels) if ridge else np.empty((0, 2)),
            annotations={series: (pa if np.isfinite(pa) else None, ridge[-1]["radius_mas"] if ridge else None)},
        )


real_example = read_mojave_example(REAL_EXAMPLE_FILENAME, REAL_EXAMPLE_SERIES)
large_samples = [real_example, *synthetic_samples]
samples = large_samples[:4]

In [ ]:
def panel_selected(sample):
    return ANNOTATED_PANELS is None or sample["id"] in ANNOTATED_PANELS


def panel_values(sample):
    if not panel_selected(sample):
        return {}
    return {
        key: sample["annotations"][key]
        for key in ACTIVE_SERIES
        if key in sample["annotations"]
    }


def panel_title(sample, *, source=False):
    if source and SHOW_SOURCE_NAMES:
        return f"{sample['id']}  |  {sample['source']}"
    return sample["id"]


def set_view(ax):
    ax.set(xlim=(VIEW_MAS, -VIEW_MAS), ylim=(-VIEW_MAS, VIEW_MAS))


def single_panel(index, *, overlays=None):
    sample = samples[index]
    figure = Figure(figsize=(5.5, 5.1), layout="constrained")
    ax = figure.subplots()
    plotting.plot_image(
        sample["image"], ax=ax, coordinates=sample["coordinates"], overlays=overlays,
        show_axes=SHOW_AXES, limits=INTENSITY_LIMITS, colorbar=True, colorbar_label="Jy/beam",
    )
    ax.set_title(panel_title(sample, source=True), loc="left")
    set_view(ax)
    return figure, ax


def preview_grid(
    records=None, *, overlays=None, row_labels=None, sources=False, ncols=2,
    show_axes=None, shared_colorbar=None, panel_labels=None,
):
    records = samples if records is None else records
    show_axes = SHOW_AXES if show_axes is None else show_axes
    shared_colorbar = SHARED_COLORBAR if shared_colorbar is None else shared_colorbar
    if panel_labels is None:
        panel_labels = [panel_title(sample, source=sources) for sample in records]
    figure, axes = plotting.plot_grid(
        [sample["image"] for sample in records], nrow=ncols,
        sample_labels=panel_labels,
        coordinates=[sample["coordinates"] for sample in records], overlays=overlays,
        show_axes=show_axes, shared_scale=shared_colorbar,
        limits=INTENSITY_LIMITS if shared_colorbar else None,
        colorbar_label="Jy/beam", row_labels=row_labels,
    )
    width = max(axes.shape[1] * 3.4 + 0.4, 6.5)
    figure.set_size_inches(width + (0.8 if row_labels else 0) + (0.7 if not shared_colorbar else 0),
                          axes.shape[0] * 3.2 + 0.3)
    for ax in list(axes.flat)[:len(records)]:
        set_view(ax)
    return figure, axes


def set_tile_caption(ax, lines):
    if lines:
        label = ax.get_xlabel()
        ax.set_xlabel("\n".join([label, *lines] if label else lines), linespacing=1.45, labelpad=7)


def show_figure(figure):
    png = BytesIO()
    figure.savefig(png, format="png", dpi=120)
    display(Image(data=png.getvalue()))


def pa_vector(pa_deg, length):
    angle = np.deg2rad(pa_deg)
    return length * np.sin(angle), length * np.cos(angle)

In [ ]:
@dataclass(frozen=True)
class OverlayNote:
    kind: str
    series: str
    label: str
    value: str
    mark: object
    marker: str = "."
    faded: bool = False


NOTE_STYLES = {
    **SERIES,
    "ridge": RIDGE_STYLE,
    "dots": dict(label="Dots", short="Dots", color="#C98DE1", linestyle="solid"),
    "crosses": dict(label="Crosses", short="Crosses", color="#8AE2B2", linestyle="solid"),
}


def panel_notes(sample, *, uniform=False, fade_arcs=None):
    if not panel_selected(sample):
        return []
    notes = []
    values = panel_values(sample)
    fade_arcs = FADE_EXTENT_ARCS if fade_arcs is None else fade_arcs
    for key in ACTIVE_SERIES if uniform else values:
        pa, extent = values.get(key, (None, None))
        style = {name: SERIES[key][name] for name in ("color", "linestyle")}
        if pa is not None or uniform:
            arrow = None if pa is None else ImageOverlay.arrow(
                (0, 0), pa_vector(pa, 1.65), linewidth=1.25, **style,
            )
            notes.append(OverlayNote("arrow", key, "PA", "—" if pa is None else f"{pa:.0f}°", arrow))
        if extent is not None or uniform:
            arc = None if pa is None or extent is None else ImageOverlay.arc(
                (0, 0), extent, angle_deg=90 - pa, delta_angle_deg=EXTENT_ARC_HALF_WIDTH_DEG,
                fade=fade_arcs, linewidth=1.25, color=style["color"],
                linestyle="solid" if fade_arcs else style["linestyle"],
            )
            notes.append(OverlayNote(
                "arc", key, "Extent", "—" if extent is None else f"{extent:.2f} mas",
                arc, faded=fade_arcs,
            ))
    if SHOW_RIDGE:
        ridge = sample["ridge"]
        line = None
        if len(ridge) >= 2:
            line = ImageOverlay.polyline(
                ridge, smooth=True, linewidth=0.9, alpha=0.85,
                color=RIDGE_STYLE["color"], linestyle=RIDGE_STYLE["linestyle"],
            )
        if len(ridge) or uniform:
            notes.append(OverlayNote(
                "polyline", "ridge", "Ridgeline", f"{len(ridge)} pts" if len(ridge) else "—", line,
            ))
    return notes


def notes_to_overlay(notes, sample):
    overlay = ImageOverlay(coordinates=sample["coordinates"])
    for note in notes:
        if note.mark is not None:
            overlay.append(note.mark)
    return overlay


angles = np.linspace(0, 2 * np.pi, 26)
GLYPH_PATHS = {
    "arrow": [(2, 5), (15, 5), (11, 8), (15, 5), (11, 2)],
    "arc": [(x, 8 - 0.065 * (x - 9)**2) for x in np.linspace(2, 16, 20)],
    "ellipse": np.column_stack((9 + 6 * np.cos(angles), 5 + 3 * np.sin(angles))),
    "line": [(2, 3), (16, 7)],
    "polyline": [(2, 3), (6, 7), (11, 4), (16, 7)],
}


def glyph_box(note):
    style = NOTE_STYLES[note.series]
    area = DrawingArea(18, 10, 0, 0)
    if note.kind == "points":
        area.add_artist(Line2D(
            [9], [5], marker=note.marker, markersize=5, linestyle="none", color=style["color"],
        ))
    elif note.faded:
        points = np.asarray(GLYPH_PATHS[note.kind])
        segments = np.stack((points[:-1], points[1:]), axis=1)
        rgba = np.tile(to_rgba(style["color"]), (len(segments), 1))
        rgba[:, 3] *= (1 + np.cos(np.linspace(-np.pi, np.pi, len(segments)))) / 2
        area.add_artist(LineCollection(segments, colors=rgba, linewidths=1.25))
    else:
        linestyle = style["linestyle"]
        area.add_artist(PathPatch(
            MarkerPath(GLYPH_PATHS[note.kind]), fill=False, edgecolor=style["color"], linewidth=1.25,
            linestyle=(0, (2, 1.2)) if linestyle in ("dashed", "--") else linestyle,
        ))
    return area


def label_box(text, color, fontsize):
    return TextArea(text, textprops={"color": color, "fontsize": fontsize})


def note_row(note, text, *, color, fontsize):
    return HPacker(children=[glyph_box(note), label_box(text, color, fontsize)], align="center", pad=0, sep=2)


def short_rows(notes, *, color, show_series=False):
    rows = []
    for key in dict.fromkeys(note.series for note in notes):
        entries = [note for note in notes if note.series == key]
        if entries:
            children = [label_box(NOTE_STYLES[key]["short"], color, 7.8)] if show_series else []
            children.extend(note_row(note, note.value, color=color, fontsize=8) for note in entries)
            rows.append(HPacker(children=children, align="center", pad=0, sep=6))
    return rows


def add_image_key(ax, notes, *, glyph_only=False, grid=False):
    if not notes:
        return
    if glyph_only:
        rows = short_rows(notes, color="white")
    else:
        rows = []
        for note in notes:
            label = f"{SERIES[note.series]['label']} {note.label}" if note.series in SERIES else note.label
            rows.append(note_row(note, f"{label}  {note.value}", color="white", fontsize=7 if grid else 8.5))
    content = VPacker(children=rows, align="left", pad=0, sep=2)
    key = AnchoredOffsetbox(loc="lower right", child=content, frameon=True, pad=0.4, borderpad=0.55)
    key.patch.set(facecolor="#172031", edgecolor="#99a5b6", linewidth=0.55, alpha=0.92)
    ax.add_artist(key)


def add_short_caption(ax, notes):
    rows = short_rows(notes, color="#26344a", show_series=True)
    if rows:
        content = VPacker(children=rows, align="left", pad=0, sep=2)
        caption = AnnotationBbox(
            content, (0.5, 0), xycoords="axes fraction",
            xybox=(0, -36 if ax.get_xlabel() else -8), boxcoords="offset points",
            box_alignment=(0.5, 1), annotation_clip=False, frameon=False, pad=0,
        )
        caption.set_clip_on(False)
        ax.add_artist(caption)


def add_grid_key(figure, panels, *, explain_glyphs=False, title=None):
    shown = {note.series for notes in panels for note in notes}
    kinds = {note.kind for notes in panels for note in notes}
    handles = []
    if explain_glyphs:
        for kind, label in (("arrow", "Jet PA"), ("arc", "Radial extent"), ("polyline", "Ridgeline")):
            if kind in kinds:
                marker = MarkerPath(np.asarray(GLYPH_PATHS[kind]) - (9, 5))
                handles.append(Line2D(
                    [], [], marker=marker, markersize=12, markerfacecolor="none",
                    markeredgecolor="#42536a", linestyle="none", label=label,
                ))
    handles.extend(
        Line2D([0, 1], [0, 0], linewidth=1.4,
               **{name: NOTE_STYLES[key][name] for name in ("label", "color", "linestyle")})
        for key in (*ACTIVE_SERIES, "ridge") if key in shown and (key != "ridge" or not explain_glyphs)
    )
    if handles:
        figure.legend(
            handles=handles, title=title, loc="outside lower center", ncols=len(handles),
            frameon=False, handlelength=2.6,
        )

## 1. Single image with a source name

The source title stands on its own. Axes are controlled by SHOW_AXES.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    figure, ax = single_panel(0)
    show_figure(figure)

## 2. Single image with a legend inside

The real observation is annotated with its measured PA, radial extent, and ridgeline. The glyphs share the color and line style of the plotted marks.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    notes = panel_notes(samples[0])
    figure, ax = single_panel(0, overlays=notes_to_overlay(notes, samples[0]))
    add_image_key(ax, notes)
    show_figure(figure)

## 3. Single image with a short strip underneath

The same arrows and arcs are drawn. The legend moves below the image, where the glyphs carry the quantity names.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    notes = panel_notes(samples[0])
    figure, ax = single_panel(0, overlays=notes_to_overlay(notes, samples[0]))
    add_short_caption(ax, notes)
    show_figure(figure)

## 4. Grid with LaTeX captions

Each tile uses the same order of series and fields. Dashes mark unavailable values. The images use plain rendering.

In [ ]:
def math_caption(pa, extent):
    angle = r"\mathrm{--}" if pa is None else rf"{pa:.0f}^\circ"
    radius = r"\mathrm{--}" if extent is None else rf"{extent:.2f}\,\mathrm{{mas}}"
    return rf"$\mathrm{{PA}}={angle}\qquad r={radius}$"


with mpl.rc_context(MATH_RC):
    figure, axes = preview_grid()
    for ax, sample in zip(axes.flat, samples):
        if panel_selected(sample):
            values = panel_values(sample)
            lines = [
                f"{SERIES[key]['label']}    {math_caption(*values.get(key, (None, None)))}"
                for key in ACTIVE_SERIES
            ]
            set_tile_caption(ax, lines)
    show_figure(figure)

## 5. Large grid with full legends inside the tiles

One real observation and eleven synthetic examples in a 3 × 4 layout. Every enabled tile uses the same list of fields and full labels. Unavailable values are shown as dashes.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    panels = [panel_notes(sample, uniform=True) for sample in large_samples]
    figure, axes = preview_grid(
        large_samples, ncols=4, sources=True,
        overlays=[notes_to_overlay(notes, sample) for notes, sample in zip(panels, large_samples)],
    )
    for ax, notes in zip(axes.flat, panels):
        add_image_key(ax, notes, grid=True)
    show_figure(figure)

## 6. Large grid with glyphs and values only

The same 3 × 4 grid and field order. The in-tile keys show only a glyph and a value. One key underneath identifies the glyphs and series.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    panels = [panel_notes(sample, uniform=True) for sample in large_samples]
    figure, axes = preview_grid(
        large_samples, ncols=4, sources=True,
        overlays=[notes_to_overlay(notes, sample) for notes, sample in zip(panels, large_samples)],
    )
    for ax, notes in zip(axes.flat, panels):
        add_image_key(ax, notes, glyph_only=True, grid=True)
    add_grid_key(figure, panels, explain_glyphs=True)
    show_figure(figure)

## 6b. Aligned labels below the tiles

Both grids keep the source titles, image overlays, and intensity scale from version 6. The first uses small glyphs in the margin. The second uses PA, r, and N for position angle, extent, and ridge point count. Each value stays in the same column, including unavailable values.

In [ ]:
def add_aligned_strip(ax, notes, *, pictograms):
    columns = {"arrow": (0.29, 0.51), "arc": (0.66, 0.93), "polyline": (0.29, 0.51)}
    symbols = {"arrow": "PA", "arc": r"$r$", "polyline": r"$N$"}
    first_row = 36 if ax.get_xlabel() else 12

    def text(value, x, y, *, align="center", color="#26344a"):
        ax.annotate(
            value, (x, 0), xycoords="axes fraction", xytext=(0, -y), textcoords="offset points",
            annotation_clip=False, clip_on=False, ha=align, va="center", fontsize=8, color=color,
        )

    for row, series in enumerate(dict.fromkeys(note.series for note in notes)):
        y = first_row + row * 12
        text(NOTE_STYLES[series]["short"], 0.04, y, align="left", color="#536074")
        for note in notes:
            if note.series != series:
                continue
            glyph_x, value_x = columns[note.kind]
            if pictograms:
                glyph = AnnotationBbox(
                    glyph_box(note), (glyph_x, 0), xycoords="axes fraction",
                    xybox=(0, -y), boxcoords="offset points", box_alignment=(0.5, 0.5),
                    annotation_clip=False, frameon=False, pad=0,
                )
                glyph.set_clip_on(False)
                ax.add_artist(glyph)
            else:
                text(symbols[note.kind], glyph_x, y, color="#536074")
            value = note.value.removesuffix(" mas").removesuffix(" pts")
            text(value, value_x, y, align="right")


with mpl.rc_context(CAPTION_RC):
    panels = [panel_notes(sample, uniform=True) for sample in large_samples]
    for pictograms in (True, False):
        figure, axes = preview_grid(
            large_samples, ncols=4, sources=True,
            overlays=[notes_to_overlay(notes, sample) for notes, sample in zip(panels, large_samples)],
        )
        figure.suptitle("Small glyphs below the tiles" if pictograms else "Scientific notation below the tiles")
        for ax, notes in zip(axes.flat, panels):
            add_aligned_strip(ax, notes, pictograms=pictograms)
        add_grid_key(
            figure, panels, explain_glyphs=pictograms,
            title="Extent in mas · Ridgeline in points" if pictograms else "PA (°) · r (mas) · N (ridge points)",
        )
        show_figure(figure)

## 7. Visible mas axes with shared and per-image colorbars

The first grid uses one scale for intensity comparisons. In the second, each tile has its own colorbar and automatic limits, which can reveal structure in fainter images. Both show east and north offsets in mas.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    for shared in (True, False):
        figure, _ = preview_grid(sources=True, show_axes=True, shared_colorbar=shared)
        figure.suptitle("Shared intensity scale" if shared else "Individual intensity scales")
        show_figure(figure)

## 8. The same final images, plain and annotated

The upper row passes None for both overlays. The lower row uses the configured overlays for the same images.

In [ ]:
with mpl.rc_context(CAPTION_RC):
    selected = samples[:2]
    panels = [panel_notes(sample, uniform=True) for sample in selected]
    figure, _ = preview_grid(
        selected * 2,
        overlays=[None, None, *[notes_to_overlay(notes, sample) for notes, sample in zip(panels, selected)]],
        row_labels=["Plain", "Annotated"], sources=True,
    )
    add_grid_key(figure, panels)
    show_figure(figure)

## 9. A plain denoising grid

These synthetic noise-to-image frames stand in for model snapshots. Sample IDs and stages provide the labels.

In [ ]:
rng = np.random.default_rng(111)
stages = []
for sample in synthetic_samples[:2]:
    image = sample["image"]
    model_image = (image - image.min()) / np.ptp(image)
    noise = rng.standard_normal(image.shape)
    stages.extend([noise, 0.35 * noise + 0.65 * model_image, model_image])

with mpl.rc_context(CAPTION_RC):
    figure, _ = plotting.plot_grid(
        stages, nrow=3, row_labels=[sample["id"] for sample in synthetic_samples[:2]],
        column_labels=["Noise", "Intermediate", "Final"],
        scale="linear", show_axes=SHOW_AXES, overlays=None, colorbar_label="Synthetic model value",
    )
    show_figure(figure)

## 10. Faded arcs and mixed marker sets

All six tiles show R01 at the same scale. The first two compare a plain arc with a faded arc at the same radius and opening angle. The other tiles vary the marks and their counts. A group of dots or crosses gets one entry showing its marker and count.

In [ ]:
def marker_note(points, *, marker, series, label, size):
    mark = None
    if len(points):
        mark = ImageOverlay.points(
            points, marker=marker, markersize=size, color=NOTE_STYLES[series]["color"],
        )
    return OverlayNote(
        "points", series, label, f"{len(points)} pts" if len(points) else "—", mark, marker=marker,
    )


ridge = real_example["ridge"]
indices = np.linspace(0, len(ridge) - 1, min(5, len(ridge)), dtype=int)
sparse = ridge[indices]
dots = marker_note(sparse, marker=".", series="dots", label="Ridge dots", size=4)
crosses = marker_note(sparse, marker="x", series="crosses", label="Ridge crosses", size=5)
dense_dots = marker_note(ridge, marker=".", series="dots", label="Ridge dots", size=3)
core = marker_note([(0, 0)], marker="x", series="crosses", label="Core", size=6)

plain = [note for note in panel_notes(real_example, fade_arcs=False) if note.kind in ("arrow", "arc")]
faded = [note for note in panel_notes(real_example, fade_arcs=True) if note.kind in ("arrow", "arc")]
mixed = [note for note in panel_notes(real_example, fade_arcs=True) if note.kind in ("arc", "polyline")]
panels = [plain, faded, [dots], [crosses], [dense_dots, core], [*mixed, crosses]]
if not panel_selected(real_example):
    panels = [[] for _ in panels]

with mpl.rc_context(CAPTION_RC):
    figure, axes = preview_grid(
        [real_example] * len(panels), ncols=3, show_axes=False,
        panel_labels=["PA and plain arc", "PA and faded arc", "Sparse dots",
                      "Sparse crosses", "Full ridge and core", "Arc, ridge and crosses"],
        overlays=[notes_to_overlay(notes, real_example) for notes in panels],
    )
    for ax, notes in zip(axes.flat, panels):
        add_image_key(ax, notes, grid=True)
    show_figure(figure)

In [ ]:
# 11. Native image with and without Stokes I contours
contour_base = 3.5 * real_example["noise_rms_jy_per_beam"]

with mpl.rc_context(CAPTION_RC):
    figure, axes = preview_grid(
        [real_example, real_example], show_axes=True, shared_colorbar=True,
        panel_labels=["Stokes I", "Stokes I with contours"],
    )
    ax = axes[0, 1]
    artist = ax.images[-1]
    brightness = artist.get_array()  # Pixel values in Jy/beam, before the display stretch.
    peak = brightness.max()
    if peak > contour_base:
        count = int(np.ceil(np.log2(peak / contour_base)))
        levels = contour_base * 2.0 ** np.arange(count)
        ax.contour(
            brightness, levels=levels, colors="white", linewidths=0.6, alpha=0.72,
            transform=artist.get_transform(),
        )

    figure.suptitle(
        f"{real_example['source']}  |  Contours start at {1_000 * contour_base:.2f} mJy/beam (×2 steps)"
    )
    show_figure(figure)